Data Wrangling


Analytical question: Is there a significant difference in the average number of interceptions per 90 minutes between defenders from teams that reached the knockout stage and defenders from teams eliminated in the group stage of the FIFA World Cup 2026?


In [1]:
import pandas as pd
import numpy as np

# First I am loading my full dataset
df = pd.read_csv(
    "WC2026_Interceptions_Analytical_datasets.csv",
    encoding="cp1252"
)

# I am checking the first few rows
print(df.head())

   Rank              Player Position   Squad  Age  Born  90s  Tackles  \
0     1    Brenden Aaronson       MF     USA   25  2000  0.8        1   
1     2      Thelo Aasgaard       MF  Norway   24  2002  1.0        1   
2     3    Hamza Abdelkarim       FW   Egypt   18  2008  0.7        0   
3     4  Hossam Abdelmaguid       DF   Egypt   25  2001  0.7        0   
4     5  Mohamed Abdelmonem       DF   Egypt   27  1999  0.2        0   

   Interceptions (Int) Tournament Stage  Interceptions per 90 mins  \
0                    0         Knockout                   0.000000   
1                    3         Knockout                   3.000000   
2                    0         Knockout                   0.000000   
3                    1         Knockout                   1.428571   
4                    0         Knockout                   0.000000   

  Eligible Defender FBref Player ID  Unnamed: 13  Unnamed: 14  Unnamed: 15  \
0                No        5bc43860          NaN          NaN 

In [2]:
# I am removing the empty unnamed columns created during the CSV export

df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# These three analytical columns already exist in the prepared CSV,
# so I am removing them and creating them again myself in Python below

columns_to_recreate = [
    "Tournament Stage",
    "Interceptions per 90 mins",
    "Eligible Defender"
]

df = df.drop(columns=columns_to_recreate, errors="ignore")

# I am checking the columns before creating my analytical variables
print(df.columns)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Index(['Rank', 'Player', 'Position', 'Squad', 'Age', 'Born', '90s', 'Tackles',
       'Interceptions (Int)', 'FBref Player ID'],
      dtype='str')
Rows: 1039
Columns: 10


In [3]:
# Now I am checking missing values in the raw variables I need for my analysis

print(df.isnull().sum())

Rank                   0
Player                 0
Position               0
Squad                  0
Age                    0
Born                   0
90s                    0
Tackles                0
Interceptions (Int)    0
FBref Player ID        0
dtype: int64


In [4]:
# I am checking for exact duplicate rows

duplicate_rows = df.duplicated().sum()
print("Exact duplicate rows:", duplicate_rows)

# I am also checking whether any FBref Player ID is repeated

duplicate_ids = df["FBref Player ID"].duplicated().sum()
missing_ids = df["FBref Player ID"].isnull().sum()

print("Duplicate FBref Player IDs:", duplicate_ids)
print("Missing FBref Player IDs:", missing_ids)

Exact duplicate rows: 0
Duplicate FBref Player IDs: 0
Missing FBref Player IDs: 0


In [5]:
# I am checking the playing-time and interception variables before calculating a per-90 rate

print("Players with 90s = 0:", (df["90s"] == 0).sum())
print("Players with negative 90s:", (df["90s"] < 0).sum())
print("Players with negative interceptions:", (df["Interceptions (Int)"] < 0).sum())

Players with 90s = 0: 33
Players with negative 90s: 0
Players with negative interceptions: 0


In [6]:
# I am defining the teams that reached the knockout stage

knockout_teams = {
    "Algeria", "Argentina", "Australia", "Austria", "Belgium", "Bosnia-Herz",
    "Brazil", "Cabo Verde", "Canada", "Colombia", "Congo DR", "Croatia",
    "Côte d'Ivoire", "Ecuador", "Egypt", "England", "France", "Germany",
    "Ghana", "Japan", "Mexico", "Morocco", "Netherlands", "Norway",
    "Paraguay", "Portugal", "Senegal", "South Africa", "Spain", "Sweden",
    "Switzerland", "USA"
}

# I am also defining the teams eliminated in the group stage

group_stage_eliminated_teams = {
    "Curaçao", "Czechia", "Haiti", "IR Iran", "Iraq", "Jordan",
    "Korea Republic", "New Zealand", "Panama", "Qatar", "Saudi Arabia",
    "Scotland", "Tunisia", "Türkiye", "Uruguay", "Uzbekistan"
}

# I am checking that every team in my dataset belongs to one of these two groups

dataset_teams = set(df["Squad"].dropna().unique())
classified_teams = knockout_teams | group_stage_eliminated_teams

unclassified_teams = sorted(dataset_teams - classified_teams)
extra_classified_teams = sorted(classified_teams - dataset_teams)

print("Unique teams in dataset:", len(dataset_teams))
print("Unclassified teams:", unclassified_teams)
print("Teams in my classification but not in dataset:", extra_classified_teams)

if unclassified_teams:
    raise ValueError("Some teams have not been assigned a tournament stage.")

Unique teams in dataset: 48
Unclassified teams: []
Teams in my classification but not in dataset: []


In [7]:
# Now I am creating the Tournament Stage column in Python

stage_map = {team: "Knockout" for team in knockout_teams}
stage_map.update({
    team: "Group Stage Eliminated"
    for team in group_stage_eliminated_teams
})

df["Tournament Stage"] = df["Squad"].map(stage_map)

# I am checking the number of teams and players in each tournament stage

print("Teams in each stage:")
print(df.groupby("Tournament Stage")["Squad"].nunique())

print("\nPlayers in each stage:")
print(df["Tournament Stage"].value_counts())

Teams in each stage:
Tournament Stage
Group Stage Eliminated    16
Knockout                  32
Name: Squad, dtype: int64

Players in each stage:
Tournament Stage
Knockout                  713
Group Stage Eliminated    326
Name: count, dtype: int64


In [8]:
# I am identifying whether each player's listed position includes defender (DF)
# The dataset stores combined positions like DFMF, MFDF and DFFW, so I am checking whether "DF" appears in Position

is_defender = df["Position"].fillna("").str.contains("DF", regex=False)

# I am calculating interceptions per 90 only when playing time is greater than zero
# This avoids dividing by zero for players who did not play

df["Interceptions per 90 mins"] = np.where(
    df["90s"] > 0,
    df["Interceptions (Int)"] / df["90s"],
    np.nan
)

# I am creating the eligible-defender variable in Python
# A player must be listed as a defender and must have positive playing time

df["Eligible Defender"] = np.where(
    is_defender & (df["90s"] > 0),
    "Yes",
    "No"
)

print(df[
    [
        "Player", "Position", "Squad", "90s", "Interceptions (Int)",
        "Tournament Stage", "Interceptions per 90 mins", "Eligible Defender"
    ]
].head(10))

               Player Position         Squad  90s  Interceptions (Int)  \
0    Brenden Aaronson       MF           USA  0.8                    0   
1      Thelo Aasgaard       MF        Norway  1.0                    3   
2    Hamza Abdelkarim       FW         Egypt  0.7                    0   
3  Hossam Abdelmaguid       DF         Egypt  0.7                    1   
4  Mohamed Abdelmonem       DF         Egypt  0.2                    0   
5            Ali Abdi       DF       Tunisia  3.0                    4   
6     Saud Abdulhamid       DF  Saudi Arabia  3.0                    2   
7   Abdulla Abdullaev       DF    Uzbekistan  2.0                    3   
8     Yusuf Abdurisag       FW         Qatar  1.1                    1   
9     Husam Abu Dahab       DF        Jordan  2.0                    0   

         Tournament Stage  Interceptions per 90 mins Eligible Defender  
0                Knockout                   0.000000                No  
1                Knockout              

In [9]:
# I am checking the per-90 calculation and defender eligibility after creating them

positive_time = df["90s"] > 0

calculation_matches = np.allclose(
    df.loc[positive_time, "Interceptions per 90 mins"],
    df.loc[positive_time, "Interceptions (Int)"] / df.loc[positive_time, "90s"],
    rtol=1e-10,
    atol=1e-12
)

print("Per-90 calculation check:", calculation_matches)
print("Eligible defenders:", (df["Eligible Defender"] == "Yes").sum())
print("Missing per-90 values:", df["Interceptions per 90 mins"].isnull().sum())

Per-90 calculation check: True
Eligible defenders: 349
Missing per-90 values: 33


In [10]:
# Now I am keeping only eligible defenders
# This removes non-defenders and defenders with no valid playing time

df_defenders = df[df["Eligible Defender"] == "Yes"].copy()

print("Eligible defenders:", df_defenders.shape[0])
print(df_defenders["Tournament Stage"].value_counts())

Eligible defenders: 349
Tournament Stage
Knockout                  238
Group Stage Eliminated    111
Name: count, dtype: int64


In [11]:
# I am checking the playing time of all eligible defenders

print(df_defenders["90s"].describe())

# I am checking how many defenders played less than one full 90-minute equivalent

low_playing_time = df_defenders[df_defenders["90s"] < 1.0]

print("\nDefenders with less than 1.0 90s:", len(low_playing_time))

print(
    low_playing_time[
        ["Player", "Squad", "90s", "Interceptions (Int)", "Interceptions per 90 mins"]
    ]
)

count    349.000000
mean       2.549570
std        1.736948
min        0.100000
25%        1.000000
50%        2.500000
75%        3.700000
max        8.300000
Name: 90s, dtype: float64

Defenders with less than 1.0 90s: 69
                     Player        Squad  90s  Interceptions (Int)  \
3        Hossam Abdelmaguid        Egypt  0.7                    1   
4        Mohamed Abdelmonem        Egypt  0.2                    0   
10    Mohammad Abu Hasheesh       Jordan  0.1                    0   
12             Mo Abualnadi       Jordan  0.8                    0   
67    Fredrik André Bjørkan       Norway  0.5                    2   
...                     ...          ...  ...                  ...   
988        Francis de Vries  New Zealand  0.2                    1   
989           Luka Vuškovi?      Croatia  0.7                    1   
1006            Axel Witsel      Belgium  0.4                    0   
1020           Manaf Younis         Iraq  0.8                    1   
1033  

In [12]:
# I am keeping defenders who played at least one full 90-minute equivalent
# I chose this rule because very small playing-time values can make per-90 rates unstable

df_final = df_defenders[df_defenders["90s"] >= 1.0].copy()

print("Defenders remaining:", df_final.shape[0])
print(df_final["Tournament Stage"].value_counts())

Defenders remaining: 280
Tournament Stage
Knockout                  190
Group Stage Eliminated     90
Name: count, dtype: int64


In [13]:
# I am checking the main analytical variable after applying my playing-time rule

print(df_final["Interceptions per 90 mins"].describe())

# I am checking one final time for duplicate player IDs and missing analysis values

print("\nDuplicate FBref Player IDs in final data:",
      df_final["FBref Player ID"].duplicated().sum())

print("Missing Interceptions per 90 values in final data:",
      df_final["Interceptions per 90 mins"].isnull().sum())

count    280.000000
mean       1.048491
std        0.734570
min        0.000000
25%        0.555556
50%        1.000000
75%        1.379870
max        5.000000
Name: Interceptions per 90 mins, dtype: float64

Duplicate FBref Player IDs in final data: 0
Missing Interceptions per 90 values in final data: 0


In [14]:
# I am putting the final columns in a clear order before saving

final_columns = [
    "Rank",
    "Player",
    "Position",
    "Squad",
    "Age",
    "Born",
    "90s",
    "Tackles",
    "Interceptions (Int)",
    "Tournament Stage",
    "Interceptions per 90 mins",
    "Eligible Defender",
    "FBref Player ID"
]

df_final = df_final[final_columns].copy()

# I am saving my cleaned defender population for the next notebook

df_final.to_csv(
    "WC2026_Cleaned_Defenders_Final.csv",
    index=False
)

print("Cleaned CSV file saved successfully.")
print("Final rows:", len(df_final))
print(df_final["Tournament Stage"].value_counts())

Cleaned CSV file saved successfully.
Final rows: 280
Tournament Stage
Knockout                  190
Group Stage Eliminated     90
Name: count, dtype: int64
